In [1]:
import confnotebook

In [2]:
from pathlib import Path

source = Path("../examples/test/bag_date/")

files = sorted(source.glob("*.pdf"))

for i, file in enumerate(files):
    print(f"[{i}] {file.stem}")

[0] 127113
[1] 127116


In [3]:
IDX_FILE = 0

In [4]:
from vision_core.debug_image_observer import DebugImageObserver

file = files[IDX_FILE]
output_dir = f"../examples/output/{file.stem}"

debug_image_observer = DebugImageObserver(output_dir=output_dir)

d:\projects\rusal_recon_srv\repo\recon_vision\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [5]:
from vision_core.pipelines.build_document import DocumentBuildPipeline

pipeline = DocumentBuildPipeline(debug_image=debug_image_observer)

document = pipeline.build(file.read_bytes())

d:\projects\rusal_recon_srv\repo\recon_vision\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', 'D:\\projects\\rusal_recon_srv\\repo\\recon_vision\\models\\PP-OCRv5_server_det')
The specified device (GPU) is not available! Switching to CPU instead.
Creating model: ('cyrillic_PP-OCRv5_mobile_rec', 'D:\\projects\\rusal_recon_srv\\repo\\recon_vision\\models\\cyrillic_PP-OCRv5_mobile_rec')
The specified device (GPU) is not available! Switching to CPU instead.
2026-06-15 09:51:11.451 | INFO     | vision_core.pipelines.build_document:build:106 - Обработка страницы 0 с dpi 200...
2026-06-15 09:51:11.595 | INFO     | vision_core.pipelines.build_document:_process_page:189 - Коррекция ориентации и наклон

In [6]:
from app.infrastructure.services.structured_data_extractor import ReconciliationActExtractor

extractor = ReconciliationActExtractor()
data = await extractor.extract(document)

2026-06-15 09:51:38.049 | DEBUG    | app.infrastructure.services.extractor.company_ext:_build_summary_text:70 - summary_text: 675 символов из 1 страниц
2026-06-15 09:51:38.050 | DEBUG    | app.infrastructure.services.extractor.company_ext:_build_summary_cell_texts:92 - summary_cell_texts: ['По данным Продавца\nО0О "ЛУКОЙЛ-Кубаньэнерго"\nинн 2312159262', 'По данным Покупателя\nАО "РУСАЛ Саяногорский\nАлюминиевый Завод\'\nинн 1902014500']
2026-06-15 09:51:38.053 | INFO     | app.infrastructure.services.extractor.company_ext:extract_companies:248 - кандидаты: ['ЛУКОЙЛ-КУБАНЬЭНЕРГО, ООО', 'РУСАЛ САЯНОГОРСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО']
2026-06-15 09:51:38.056 | DEBUG    | app.infrastructure.services.extractor.company_ext:_assign_roles:185 - события: ['ОТ ПРОДАВЦА', 'ОТ ПОКУПАТЕЛЯ']
2026-06-15 09:51:38.058 | DEBUG    | app.infrastructure.services.extractor.company_ext:_assign_roles:187 - рабочая пара: ['ЛУКОЙЛ-КУБАНЬЭНЕРГО, ООО', 'РУСАЛ САЯНОГОРСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО']
2026-06-15 09:51:38.0

AttributeError: 'NoneType' object has no attribute 'start'

In [ ]:
print(data.debit)

[LedgerEntry(record='САЛЬДО НА 01.01.2026 Г. ПО ОПЛАТЕ МОЩНОСТИ, В Т.Ч. НДС СВЕРНУТОЕ', value=65568.36, date='01.01.2026', row_reference=RowReference(id_table='0', id_row='2', id_col=1, buyer_col=3)), LedgerEntry(record='РАЗВЕРНУТОЕ', value=65568.36, date=None, row_reference=RowReference(id_table='0', id_row='3', id_col=1, buyer_col=3)), LedgerEntry(record='ПО ОПЛАТЕ НЕУСТОЙКИ (ШТРАФОВ, ПЕНИ)', value=0.0, date=None, row_reference=RowReference(id_table='0', id_row='4', id_col=1, buyer_col=3)), LedgerEntry(record='ПРИОБРЕТЕНО МОЩНОСТИ НА СУММУ, В Т.Ч. НДС', value=0.0, date=None, row_reference=RowReference(id_table='0', id_row='5', id_col=1, buyer_col=3)), LedgerEntry(record='НАЧИСЛЕНА НЕУСТОЙКА (ШТРАФЫ, ПЕНИ )', value=0.0, date=None, row_reference=RowReference(id_table='0', id_row='6', id_col=1, buyer_col=3)), LedgerEntry(record='ОПЛАЧЕНО: МОЩНОСТЬ, В Т.Ч. НДС', value=0.0, date=None, row_reference=RowReference(id_table='0', id_row='7', id_col=1, buyer_col=3)), LedgerEntry(record='НЕУСТОЙ

In [ ]:
from app.application.dto.fill_reconciliation_act import FillReconciliationActCommand
from app.domain.entities.process import ProcessState
from app.infrastructure.services.pdf_filler import DocumentPdfFiller

process_state = ProcessState(
    process_id="notebook-test",
    source_pdf=files[IDX_FILE].read_bytes(),
    document_payload=document,
)

comments = """
            По данным АО "РУСАЛ Новокузнецк" на 30.09.2023
            задолженность в пользу АО "РУСАЛ Новокузнецк"
            составляет 13 755 023,24 руб.
            С разногласиями, протокол разногласий прилагается.
            Акт сверки проверен ОУФО ОЦО, ООО "РЦУ".
            Исполнитель: Воробьева Оксана Евгеньевна
            Дата:29.01.2025
            """

# используем значения продавца для заполнения колонок покупателя
command = FillReconciliationActCommand(
    process_id="notebook-test",
    comments=comments,
    debit=data.debit,
    credit=data.credit,
)

filler = DocumentPdfFiller()
filled_pdf = await filler.fill(process_state, command)

2026-05-06 17:17:39.431 | INFO     | app.infrastructure.services.pdf_fill.render:resolve_font_file:221 - найден шрифт: /mnt/data/projects/rusal_recon_srv/repo/recon_vision/assets/fonts/LiberationSerif-Regular.ttf
2026-05-06 17:17:39.489 | DEBUG    | app.infrastructure.services.pdf_fill.render:load_aligned_page_images:49 - page=0 dpi=200 source=1654x2339 aligned=1654x2339 canvas=1654x2339
2026-05-06 17:17:39.491 | DEBUG    | app.infrastructure.services.pdf_filler:fill:48 - заполняем таблица=0 R2:C5 значение=3151089.2
2026-05-06 17:17:39.491 | DEBUG    | app.infrastructure.services.pdf_fill.render:load_font:229 - загружаем шрифт из /mnt/data/projects/rusal_recon_srv/repo/recon_vision/assets/fonts/LiberationSerif-Regular.ttf для размера 26
2026-05-06 17:17:39.493 | DEBUG    | app.infrastructure.services.pdf_filler:fill:48 - заполняем таблица=0 R3:C5 значение=0.0
2026-05-06 17:17:39.493 | DEBUG    | app.infrastructure.services.pdf_filler:fill:48 - заполняем таблица=0 R4:C5 значение=0.0
202

In [ ]:
import base64

from IPython.display import HTML, display

b64 = base64.b64encode(filled_pdf).decode()
display(HTML(f'<a href="data:application/pdf;base64,{b64}" download="filled.pdf">Скачать заполненный PDF</a>'))